# FarmFederate: Tea Leaf Disease Detection
This notebook runs the complete FarmFederate multimodal architecture (LLM + ViT + VLM + FedAvg) on the Real Dataset.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Tea Leaf Disease Detection — Standalone Real Dataset Script
============================================================
Fully self-contained script using the FarmFederate architecture
(LLM + ViT + VLM + FedAvg) on ONLY the Real Dataset.

Pipeline:
  1. Read Real Dataset/images + Real Dataset/labels (YOLO OBB)
  2. Extract & classify 371 disease crops into 5 classes
  3. Generate a text description per crop (literature-grounded symptoms)
  4. Train LLM (text), ViT (image crops), VLM (text+image) + FedAvg

Usage:
    python backend/tea_real_dataset_train.py
    python backend/tea_real_dataset_train.py --epochs 20 --fed_rounds 10
"""
from __future__ import annotations

import argparse, copy, io as _io, math, os, random, sys, warnings, json
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score, recall_score)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from tqdm import tqdm

warnings.filterwarnings("ignore")
if hasattr(sys.stdout, "buffer") and sys.stdout.encoding and sys.stdout.encoding.lower() != "utf-8":
    sys.stdout = _io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")

# ═══════════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════════════════════
LABELS = ["gray_blight", "helopeltis", "algal_leaf_spot", "brown_blight", "red_leaf_spot"]
NUM_CLS = 5
OBB_MAP = {0: "gray_blight", 1: "helopeltis", 2: "algal_leaf_spot",
           3: "brown_blight", 4: "red_leaf_spot"}

@dataclass
class Cfg:
    batch_size: int = 16; epochs: int = 15; lr: float = 1e-4
    weight_decay: float = 0.01; patience: int = 6; warmup: float = 0.05
    accum: int = 2; amp: bool = True
    num_clients: int = 3; fed_rounds: int = 8; local_epochs: int = 3
    dirichlet_alpha: float = 1.0
    img_size: int = 224; crop_pad: float = 0.10
    train_split: float = 0.80; val_split: float = 0.10
    max_seq: int = 128; seed: int = 42
    data_dir: str = "C:/Users/USER_HP/Desktop/FarmFederate/Real Dataset"
    out_dir: str  = "C:/Users/USER_HP/Desktop/FarmFederate/tea_results"

# ═══════════════════════════════════════════════════════════════════════════════
# TEXT GENERATION (symptom vocabulary from Tea Literature papers)
# ═══════════════════════════════════════════════════════════════════════════════
_KW = {
    0: {"obs": ["grey to brown blotches on mature leaves",
                "grayish discoloration under humid conditions",
                "lesions with lighter centers and darker margins",
                "grey mycelium visible on aging lesions"],
        "sym": ["grey-brown lesions on both leaf surfaces",
                "pale gray centers with dark brown borders",
                "blighted areas with grey powdery coating",
                "necrotic spots with irregular grey-brown coloration",
                "faded discoloration covering significant leaf area"],
        "cnd": ["high humidity above 85% sustained for days",
                "dense canopy reducing light penetration",
                "rain splash dispersing spores between bushes"],
        "ind": ["lesion area exceeding 30% of leaf lamina",
                "sporulation visible on aged lesion surface"]},
    1: {"obs": ["insect puncture marks visible under magnification",
                "feeding scars from tea mosquito bug on young shoots",
                "wilting of terminal bud after pest feeding"],
        "sym": ["pinhole-size spots enlarging over time",
                "sunken dark lesions with angular shape",
                "apical bud blackening after helopeltis attack",
                "chlorotic haloes around feeding punctures"],
        "cnd": ["warm dry periods increasing pest activity",
                "young flush attracting feeding insects",
                "shaded patches with high pest pressure"],
        "ind": ["population count above economic threshold",
                "damage incidence exceeding 10% of new shoots"]},
    2: {"obs": ["green to brownish-green blotches on adaxial surface",
                "algal growth forming raised velvety spots",
                "circular green-orange patches reducing photosynthesis"],
        "sym": ["small circular green spots with orange tinge",
                "velvety algal growth on leaf surface",
                "raised circular patches green center fading to brown",
                "rough felt-like circular patches scattered on blade"],
        "cnd": ["high humidity and abundant light enabling algal growth",
                "excess moisture from frequent rain",
                "old bushes with rough bark providing inoculum"],
        "ind": ["multiple spots reducing photosynthetic capacity",
                "incidence above 20% of leaf surface"]},
    3: {"obs": ["lesions on both surfaces with browning patterns",
                "tissue death spreading from margins inward",
                "dark necrotic patches along leaf edges",
                "necrosis spreading from tip toward midrib"],
        "sym": ["small brown to dark lesions with defined borders",
                "tan centers with dark margins",
                "scattered dark spots merging into large necrotic areas",
                "circular to irregular dark patches on lamina"],
        "cnd": ["high humidity and warm temperatures favoring fungal growth",
                "dense plucking table with poor air circulation",
                "spore dispersal active during monsoon season"],
        "ind": ["severity index above 25% of leaf area",
                "infection spreading to multiple shoots"]},
    4: {"obs": ["yellowing preceding visible red lesion formation",
                "scarlet to brown spots enlarging on surface",
                "red-brown spots scattered across both surfaces"],
        "sym": ["small red to dark brown lesions with yellow halo",
                "scarlet spots developing into irregular brown patches",
                "dark red circular spots with pale centers",
                "multiple small red dots merging into necrotic zone"],
        "cnd": ["warm wet conditions promoting sporulation",
                "leaf wetness from dew facilitating infection",
                "high nitrogen producing susceptible flush"],
        "ind": ["lesion count above five per leaf",
                "red spot incidence exceeding 15% of sample"]},
}

REMEDY_RECOMMENDATIONS = {
    "gray_blight": "Prune severely affected branches. Apply copper-based fungicides (e.g., Copper Oxychloride) during early symptomatic stages. Maintain good drainage and reduce shade.",
    "helopeltis": "Apply systemic insecticides like Thiamethoxam or Imidacloprid. Remove alternate hosts. Pluck shoots regularly to remove eggs. Use light traps for adult monitoring.",
    "algal_leaf_spot": "Improve air circulation and sunlight penetration through pruning. Apply Bordeaux mixture or Copper formulations. Ensure balanced NPK fertilization to improve plant vigor.",
    "brown_blight": "Remove and destroy infected leaves. Spray protective fungicides like Mancozeb before the monsoon. Maintain bush hygiene and optimize shade levels to reduce humidity.",
    "red_leaf_spot": "Ensure proper plucking cycles. Apply appropriate fungicides if severity exceeds 10%. Avoid excessive nitrogen application without balancing with potassium.",
}

_TMPLS = [
    "FIELD OBSERVATION: {obs}. Symptoms: {s1} and {s2}. Conditions: {cnd}. Assessment: {ind}.",
    "CROP REPORT: {s1} noted along with {s2}. Context: {cnd}. {obs}. Status: {ind}.",
    "AGRONOMIC SURVEY: Primary: {s1}. Secondary: {s2}. Background: {obs}. Status: {ind}.",
    "DIAGNOSTIC REPORT: Signs include {s1} with {s2}. {cnd}. Reading: {ind}.",
    "SCOUTING REPORT: {obs}. Leaf shows {s1} and {s2}. {cnd}. Risk: {ind}.",
]

def gen_crop_descriptions(samples, seed=42):
    """Generate one text description per OBB crop based on its disease class.

    Each crop from the Real Dataset gets a unique, literature-grounded text
    description derived from its classified disease.  This produces matched
    (image, text) pairs for multimodal training.

    Args:
        samples: list of (image_path, class_id, obb_coords) from OBBDS
        seed: random seed for reproducibility

    Returns:
        DataFrame with columns: text, labels, label_name  (1 row per crop)
    """
    rng = random.Random(seed)
    texts, labels, names = [], [], []
    for img_path, cid, coords in samples:
        kw = _KW[cid]; t = rng.choice(_TMPLS)
        o, s1 = rng.choice(kw["obs"]), rng.choice(kw["sym"])
        s2 = rng.choice([s for s in kw["sym"] if s != s1])
        c, ind = rng.choice(kw["cnd"]), rng.choice(kw["ind"])
        # Add crop-specific context (source image name)
        crop_ctx = f" Source: {Path(img_path).stem}, region {coords[0]:.2f},{coords[1]:.2f}."
        texts.append(t.format(obs=o, s1=s1, s2=s2, cnd=c, ind=ind) + crop_ctx)
        labels.append(cid)
        names.append(LABELS[cid])
    return pd.DataFrame({"text": texts, "labels": labels, "label_name": names})

# ═══════════════════════════════════════════════════════════════════════════════
# TOKENIZER
# ═══════════════════════════════════════════════════════════════════════════════
class Tok:
    def __init__(self, vs=30522):
        self.vs, self.pad, self.cls_id, self.sep = vs, 0, 101, 102
    def __call__(self, text, max_length=128, **kw):
        ids = [self.cls_id] + [(hash(w) % (self.vs-104))+104
               for w in text.lower().split()] + [self.sep]
        if len(ids) > max_length: ids = ids[:max_length-1] + [self.sep]
        am = [1]*len(ids)
        pad = max_length - len(ids)
        if pad > 0: ids += [self.pad]*pad; am += [0]*pad
        return {"input_ids": torch.tensor([ids], dtype=torch.long),
                "attention_mask": torch.tensor([am], dtype=torch.long)}

_tok = Tok()

# ═══════════════════════════════════════════════════════════════════════════════
# DATASETS
# ═══════════════════════════════════════════════════════════════════════════════
class TextDS(Dataset):
    def __init__(self, df, mx=128):
        self.df, self.mx = df.reset_index(drop=True), mx
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]; enc = _tok(str(r["text"]), max_length=self.mx)
        lbl = torch.zeros(NUM_CLS); ll = r["labels"]
        for l in (ll if isinstance(ll, list) else [ll]):
            if 0 <= l < NUM_CLS: lbl[l] = 1.0
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0), "labels": lbl}

class OBBDS(Dataset):
    def __init__(self, img_dir, lbl_dir, tf=None, pad=0.1, idx=None):
        self.tf, self.pad = tf, pad; self.samples = []
        for lf in sorted(Path(lbl_dir).glob("*.txt")):
            ip = Path(img_dir) / (lf.stem + ".jpg")
            if not ip.exists(): continue
            for line in lf.read_text().splitlines():
                p = line.strip().split()
                if len(p) < 9: continue
                d = OBB_MAP.get(int(p[0]));
                if d is None: continue
                self.samples.append((ip, LABELS.index(d), [float(v) for v in p[1:9]]))
        if idx is not None: self.samples = [self.samples[i] for i in idx]
    def _crop(self, img, c):
        W, H = img.size
        xs = [c[i]*W for i in range(0,8,2)]; ys = [c[i]*H for i in range(1,8,2)]
        px, py = self.pad*W, self.pad*H
        x0, y0 = max(0, min(xs)-px), max(0, min(ys)-py)
        x1, y1 = min(W, max(xs)+px), min(H, max(ys)+py)
        if x1-x0 < 4: x0, x1 = max(0, x0-8), min(W, x1+8)
        if y1-y0 < 4: y0, y1 = max(0, y0-8), min(H, y1+8)
        return img.crop((x0, y0, x1, y1))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        ip, cid, cn = self.samples[i]
        crop = self._crop(Image.open(ip).convert("RGB"), cn)
        if self.tf: crop = self.tf(crop)
        return {"pixel_values": crop, "labels": torch.tensor(cid, dtype=torch.long)}
    @property
    def labels(self): return [s[1] for s in self.samples]

class MMDS(Dataset):
    def __init__(self, obb, tdf, tf=None, mx=128, seed=42):
        self.obb, self.tf, self.mx = obb, tf, mx
        rng = random.Random(seed)
        ct = {i: [] for i in range(NUM_CLS)}
        for _, r in tdf.iterrows():
            l = r["labels"][0] if isinstance(r["labels"], list) else int(r["labels"])
            if 0 <= l < NUM_CLS: ct[l].append(str(r["text"]))
        self._ta = [rng.choice(ct.get(s[1], [])) if ct.get(s[1]) else f"Disease {s[1]}"
                    for s in obb.samples]
    def __len__(self): return len(self.obb)
    def __getitem__(self, i):
        ip, cid, cn = self.obb.samples[i]
        crop = self.obb._crop(Image.open(ip).convert("RGB"), cn)
        if self.tf: crop = self.tf(crop)
        enc = _tok(self._ta[i], max_length=self.mx)
        lbl = torch.zeros(NUM_CLS); lbl[cid] = 1.0
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "pixel_values": crop, "labels": lbl}

# ═══════════════════════════════════════════════════════════════════════════════
# TRANSFORMS & UTILS
# ═══════════════════════════════════════════════════════════════════════════════
_mean, _std = [0.485,0.456,0.406], [0.229,0.224,0.225]
def aug_tf(sz): return T.Compose([T.Resize((sz+32,sz+32)), T.RandomCrop(sz),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(0.3), T.RandomRotation(20),
    T.ColorJitter(0.3,0.3,0.3,0.05), T.ToTensor(), T.Normalize(_mean,_std)])
def val_tf(sz): return T.Compose([T.Resize((sz,sz)), T.ToTensor(), T.Normalize(_mean,_std)])

def class_weights(labels, nc):
    c = Counter(labels)
    f = torch.tensor([c.get(i,1) for i in range(nc)], dtype=torch.float)
    w = torch.sqrt(1.0/f); w = w/w.sum()*nc; return torch.clamp(w/w.mean(), max=10.0)

class BalSampler:
    def __init__(self, labels, bs=16, nc=5):
        self.bs, self.nc = bs, nc
        self.ci = {i: [] for i in range(nc)}
        for j, l in enumerate(labels):
            ll = l[0] if isinstance(l,(list,tuple)) else int(l)
            if 0 <= ll < nc: self.ci[ll].append(j)
        self.spc = max(1, bs // nc)
        self.nb = max(1, max((len(v) for v in self.ci.values() if v), default=1) // self.spc)
    def __iter__(self):
        sh = {}
        for k, v in self.ci.items():
            s = v.copy(); random.shuffle(s)
            need = self.nb * self.spc
            if len(s) < need and s: s = (s * ((need//len(s))+1))[:need]
            sh[k] = s
        pt = {i: 0 for i in range(self.nc)}
        for _ in range(self.nb):
            b = []
            for k in range(self.nc):
                for _ in range(self.spc):
                    if not sh[k]: continue
                    p = pt[k] % len(sh[k]); b.append(sh[k][p]); pt[k] = p+1
            random.shuffle(b); yield b[:self.bs]
    def __len__(self): return self.nb

# ═══════════════════════════════════════════════════════════════════════════════
# MODELS (FarmFederate architecture)
# ═══════════════════════════════════════════════════════════════════════════════
class DivLoss(nn.Module):
    def __init__(self, nc=5, w=1.0):
        super().__init__(); self.w, self.me = w, math.log(nc)
    def forward(self, logits):
        p = F.softmax(logits, -1).mean(0)
        e = -torch.sum(p * torch.log(p + 1e-8))
        return self.w * (1.0 - e / self.me)

class LLM(nn.Module):
    """Lightweight Transformer text classifier — FarmFederate architecture."""
    def __init__(self, nc=NUM_CLS, vs=30522, dim=256, mx=128, drop=0.3):
        super().__init__(); self.nc = nc
        self.emb = nn.Embedding(vs, dim); self.pos = nn.Embedding(mx, dim)
        self.ln = nn.LayerNorm(dim)
        el = nn.TransformerEncoderLayer(d_model=dim, nhead=8, dim_feedforward=dim*4,
                                        dropout=drop, batch_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(el, num_layers=4)
        self.pool = nn.AdaptiveAvgPool1d(1); self.pn = nn.LayerNorm(dim)
        self.head = nn.Sequential(nn.Linear(dim,dim), nn.GELU(), nn.Dropout(drop),
                                  nn.Linear(dim,128), nn.GELU(), nn.Dropout(drop*0.5),
                                  nn.Linear(128, nc))
    def forward(self, input_ids, attention_mask=None, labels=None):
        B, S = input_ids.shape
        x = self.emb(input_ids) + self.pos(torch.arange(S, device=input_ids.device).unsqueeze(0).expand(B,-1))
        x = self.ln(x)
        mask = (attention_mask == 0) if attention_mask is not None else None
        x = self.enc(x, src_key_padding_mask=mask)
        x = self.pn(self.pool(x.transpose(1,2)).squeeze(-1))
        logits = self.head(x)
        loss = None
        if labels is not None:
            t = labels.argmax(-1) if labels.dim()>1 and labels.size(-1)>1 else labels.squeeze(-1) if labels.dim()>1 else labels
            loss = F.cross_entropy(logits, t.long(), label_smoothing=0.2)
        return {"loss": loss, "logits": logits}

class ViT(nn.Module):
    """Residual CNN vision classifier — FarmFederate architecture."""
    def __init__(self, nc=NUM_CLS, cw=None, focal=False, gamma=2.0, ls=0.1):
        super().__init__(); self.nc, self.focal, self.gamma, self.ls = nc, focal, gamma, ls
        if cw is not None: self.register_buffer("cw", cw)
        else: self.cw = None
        self.stem = nn.Sequential(nn.Conv2d(3,64,7,2,3), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(3,2,1))
        self.b1 = nn.Sequential(nn.Conv2d(64,128,3,1,1), nn.BatchNorm2d(128), nn.ReLU(), nn.Dropout2d(0.1),
                                nn.Conv2d(128,128,3,1,1), nn.BatchNorm2d(128), nn.ReLU())
        self.d1 = nn.Conv2d(64,128,1)
        self.b2 = nn.Sequential(nn.Conv2d(128,256,3,1,1), nn.BatchNorm2d(256), nn.ReLU(), nn.Dropout2d(0.1),
                                nn.Conv2d(256,256,3,1,1), nn.BatchNorm2d(256), nn.ReLU())
        self.d2 = nn.Conv2d(128,256,1)
        self.b3 = nn.Sequential(nn.Conv2d(256,512,3,1,1), nn.BatchNorm2d(512), nn.ReLU(), nn.Dropout2d(0.15),
                                nn.Conv2d(512,512,3,1,1), nn.BatchNorm2d(512), nn.ReLU())
        self.d3 = nn.Conv2d(256,512,1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.LayerNorm(512),
                                  nn.Linear(512,256), nn.GELU(), nn.Dropout(0.3), nn.Linear(256,nc))
    def forward(self, pixel_values, labels=None):
        x = self.stem(pixel_values)
        x = self.b1(x)+self.d1(x); x = self.b2(x)+self.d2(x); x = self.b3(x)+self.d3(x)
        logits = self.head(self.pool(x))
        loss = None
        if labels is not None:
            t = labels.argmax(-1) if labels.dim()>1 and labels.size(-1)>1 else labels.squeeze(-1) if labels.dim()>1 else labels
            t = t.long()
            if self.focal:
                p = F.softmax(logits,-1); pt = p[torch.arange(len(t),device=logits.device),t]
                ce = F.cross_entropy(logits,t,reduction="none",label_smoothing=self.ls)
                fw = (1-pt)**self.gamma
                if self.cw is not None: fw = self.cw[t]*fw
                loss = (fw*ce).mean()
            elif self.cw is not None: loss = F.cross_entropy(logits,t,weight=self.cw,label_smoothing=self.ls)
            else: loss = F.cross_entropy(logits,t,label_smoothing=self.ls)
        return {"loss": loss, "logits": logits}

class VLM(nn.Module):
    """Concat-fusion multimodal — FarmFederate architecture."""
    def __init__(self, nc=NUM_CLS, td=256, vd=512, drop=0.3, ls=0.1, cw=None, focal=False, gamma=2.0):
        super().__init__(); self.nc, self.ls, self.focal, self.gamma = nc, ls, focal, gamma
        if cw is not None: self.register_buffer("cw", cw)
        else: self.cw = None
        self.te = nn.Embedding(30522, td)
        self.tenc = nn.TransformerEncoderLayer(d_model=td, nhead=4, dim_feedforward=td*4,
                                               dropout=drop, batch_first=True)
        self.tp = nn.AdaptiveAvgPool1d(1); self.td_drop = nn.Dropout(drop)
        self.ve = nn.Sequential(nn.Conv2d(3,64,7,2,3), nn.BatchNorm2d(64), nn.ReLU(), nn.Dropout2d(drop*0.5),
                                nn.MaxPool2d(3,2,1),
                                nn.Conv2d(64,128,3,1,1), nn.BatchNorm2d(128), nn.ReLU(), nn.Dropout2d(drop*0.5),
                                nn.Conv2d(128,256,3,1,1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d((7,7)))
        self.vp = nn.Linear(256*7*7, vd); self.vd = nn.Dropout(drop)
        self.head = nn.Sequential(nn.LayerNorm(td+vd), nn.Dropout(drop),
                                  nn.Linear(td+vd,256), nn.GELU(), nn.Dropout(drop), nn.Linear(256,nc))
    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        tx = self.td_drop(self.tp(self.tenc(self.te(input_ids)).transpose(1,2)).squeeze(-1))
        vx = self.vd(self.vp(self.ve(pixel_values).flatten(1)))
        logits = self.head(torch.cat([tx, vx], -1))
        loss = None
        if labels is not None:
            t = labels.argmax(-1) if labels.dim()>1 and labels.size(-1)>1 else labels.squeeze(-1) if labels.dim()>1 else labels
            t = t.long()
            if self.focal:
                p = F.softmax(logits,-1); pt = p[torch.arange(len(t),device=logits.device),t]
                ce = F.cross_entropy(logits,t,reduction="none",label_smoothing=self.ls)
                fw = (1-pt)**self.gamma
                if self.cw is not None: fw = self.cw[t]*fw
                loss = (fw*ce).mean()
            elif self.cw is not None: loss = F.cross_entropy(logits,t,weight=self.cw,label_smoothing=self.ls)
            else: loss = F.cross_entropy(logits,t,label_smoothing=self.ls)
        return {"loss": loss, "logits": logits}

# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING
# ═══════════════════════════════════════════════════════════════════════════════
def warmup_cos(opt, ws, ts):
    def f(s):
        if s < ws: return s / max(1, ws)
        p = (s - ws) / max(1, ts - ws)
        return max(0.1, 0.5*(1+np.cos(np.pi*p)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def evaluate(model, dl, dev, mt="vision"):
    model.eval(); preds, lbls = [], []
    with torch.no_grad():
        for b in dl:
            b = {k: v.to(dev) if isinstance(v, torch.Tensor) else v for k, v in b.items()}
            if mt == "text": o = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
            elif mt == "vision": o = model(pixel_values=b["pixel_values"])
            else: o = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"], pixel_values=b["pixel_values"])
            preds.append(o["logits"].argmax(-1).cpu())
            l = b["labels"]
            if l.dim()>1 and l.size(-1)>1: l = l.argmax(-1)
            elif l.dim()>1: l = l.squeeze(-1)
            lbls.append(l.cpu())
    p, l = torch.cat(preds).numpy(), torch.cat(lbls).numpy()
    return {"f1": f1_score(l,p,average="micro",zero_division=0),
            "f1_macro": f1_score(l,p,average="macro",zero_division=0),
            "acc": accuracy_score(l,p), "preds": p, "labels": l,
            "f1_cls": f1_score(l,p,average=None,zero_division=0,labels=list(range(NUM_CLS))).tolist()}

def train_model(model, tr_dl, va_dl, cfg, dev, mt="vision", models_dir=None, prefix=""):
    lr = max(cfg.lr, {"text":1e-4,"vision":1e-4,"multimodal":8e-5}.get(mt,1e-4))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg.weight_decay)
    ts = len(tr_dl)*cfg.epochs; sch = warmup_cos(opt, max(1,int(0.05*ts)), ts)
    use_amp = dev.type=="cuda" and cfg.amp
    scaler = torch.amp.GradScaler("cuda") if use_amp else None
    div_fn = DivLoss(NUM_CLS)
    best_f1, best_st, hist = 0.0, None, {"loss":[], "f1":[]}
    pat = 0
    for ep in range(cfg.epochs):
        model.train(); tl = 0; opt.zero_grad()
        for bi, b in enumerate(tqdm(tr_dl, desc=f"Ep{ep+1}", leave=False)):
            b = {k: v.to(dev) if isinstance(v, torch.Tensor) else v for k, v in b.items()}
            with torch.amp.autocast("cuda", enabled=use_amp):
                if mt=="text": o = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"], labels=b["labels"])
                elif mt=="vision": o = model(pixel_values=b["pixel_values"], labels=b["labels"])
                else: o = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"], pixel_values=b["pixel_values"], labels=b["labels"])
                loss = (o["loss"] + div_fn(o["logits"])) / cfg.accum
            (scaler.scale(loss) if use_amp else loss).backward()
            if (bi+1)%cfg.accum==0 or bi+1==len(tr_dl):
                if use_amp: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                (scaler.step(opt) if use_amp else opt.step())
                if use_amp: scaler.update()
                sch.step(); opt.zero_grad()
            tl += o["loss"].item()
        tl /= max(len(tr_dl),1); m = evaluate(model, va_dl, dev, mt)
        hist["loss"].append(tl); hist["f1"].append(m["f1"])
        
        # Save per-epoch checkpoint
        if models_dir:
            torch.save({"state_dict": model.state_dict(), "class": model.__class__.__name__,
                        "labels": LABELS}, models_dir / f"cent_{prefix}_ep{ep+1}.pt")
                        
        tag = ""
        if m["f1"] > best_f1:
            best_f1 = m["f1"]; best_st = {k:v.cpu().clone() for k,v in model.state_dict().items()}
            pat = 0; tag = " *"
        else: pat += 1
        print(f"  Ep {ep+1}/{cfg.epochs} [{mt}] Loss={tl:.4f} F1={m['f1']:.4f} Acc={m['acc']:.4f}{tag}")
        if pat >= cfg.patience: print("  Early stopping."); break
    if best_st: model.load_state_dict(best_st)
    return best_f1, hist, best_st

def federated_train(m_cls, m_kw, dataset, va_dl, cfg, dev, mt="vision", models_dir=None, prefix=""):
    N = len(dataset); K = cfg.num_clients
    # Dirichlet split
    labels = [dataset[i]["labels"] for i in range(N)]
    flat = [l.argmax().item() if l.dim()>0 and l.numel()>1 else int(l) for l in labels]
    ci = {c: [i for i,l in enumerate(flat) if l==c] for c in range(NUM_CLS)}
    splits = [[] for _ in range(K)]
    for c, idxs in ci.items():
        props = np.random.dirichlet([cfg.dirichlet_alpha]*K)
        random.shuffle(idxs); ptr = 0
        for k in range(K):
            n = max(1, int(len(idxs)*props[k]))
            splits[k].extend(idxs[ptr:ptr+n]); ptr += n
        if ptr < len(idxs): splits[0].extend(idxs[ptr:])

    global_model = m_cls(**m_kw).to(dev); gs = global_model.state_dict()
    best_f1, hist = 0.0, {"rounds":[], "f1":[]}
    for rnd in range(cfg.fed_rounds):
        print(f"  Fed round {rnd+1}/{cfg.fed_rounds}")
        states = []
        for k in range(K):
            local = m_cls(**m_kw).to(dev); local.load_state_dict(gs)
            sub = torch.utils.data.Subset(dataset, splits[k])
            dl = DataLoader(sub, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
            opt = torch.optim.AdamW(local.parameters(), lr=cfg.lr)
            for _ in range(cfg.local_epochs):
                local.train()
                for b in dl:
                    b = {k2: v.to(dev) if isinstance(v, torch.Tensor) else v for k2,v in b.items()}
                    if mt=="text": o = local(input_ids=b["input_ids"], attention_mask=b["attention_mask"], labels=b["labels"])
                    elif mt=="vision": o = local(pixel_values=b["pixel_values"], labels=b["labels"])
                    else: o = local(input_ids=b["input_ids"], attention_mask=b["attention_mask"], pixel_values=b["pixel_values"], labels=b["labels"])
                    o["loss"].backward(); opt.step(); opt.zero_grad()
            states.append({k2: v.cpu() for k2,v in local.state_dict().items()})
        # FedAvg
        avg = {}
        for key in gs:
            avg[key] = torch.stack([s[key].float() for s in states]).mean(0)
        global_model.load_state_dict(avg); gs = global_model.state_dict()
        
        m = evaluate(global_model, va_dl, dev, mt)
        hist["rounds"].append(rnd+1); hist["f1"].append(m["f1"])
        
        if models_dir:
            torch.save({"state_dict": global_model.state_dict(), "class": m_cls.__name__,
                        "labels": LABELS}, models_dir / f"fed_{prefix}_rd{rnd+1}.pt")
                        
        tag = " (best)" if m["f1"] > best_f1 else ""
        if m["f1"] > best_f1: best_f1 = m["f1"]
        print(f"    Global F1={m['f1']:.4f}{tag}")
    return best_f1, hist

# ═══════════════════════════════════════════════════════════════════════════════
# PLOTS
# ═══════════════════════════════════════════════════════════════════════════════
def save_plots(results, out_dir):
    pd_dir = Path(out_dir) / "plots"; pd_dir.mkdir(parents=True, exist_ok=True)
    try: plt.style.use('seaborn-v0_8-whitegrid')
    except: plt.style.use('ggplot')

    # 1. Model comparison bar
    fig, ax = plt.subplots(figsize=(10,5))
    names = [f"{k.upper()} (cent)" for k in results] + [f"{k.upper()} (fed)" for k in results]
    vals = [results[k]["cent_f1"] for k in results] + [results[k]["fed_f1"] for k in results]
    colors = ['#3498db']*3 + ['#e74c3c']*3
    ax.barh(names, vals, color=colors, edgecolor='black')
    ax.set_xlabel("F1 Score"); ax.set_title("Centralised vs Federated — Real Dataset")
    ax.set_xlim(0,1); plt.tight_layout(); fig.savefig(pd_dir/"comparison.png", dpi=200); plt.close()

    # 2. Training curves
    for k in results:
        if "history" not in results[k]: continue
        h = results[k]["history"]
        fig, (a1,a2) = plt.subplots(1,2, figsize=(12,4))
        a1.plot(h["loss"], 'b-o'); a1.set_title(f"{k.upper()} Train Loss"); a1.set_xlabel("Epoch")
        a2.plot(h["f1"], 'g-o'); a2.set_title(f"{k.upper()} Val F1"); a2.set_xlabel("Epoch")
        plt.tight_layout(); fig.savefig(pd_dir/f"curves_{k}.png", dpi=200); plt.close()

    # 3. Confusion matrices
    for k in results:
        if "test_preds" not in results[k]: continue
        cm = confusion_matrix(results[k]["test_labels"], results[k]["test_preds"], labels=list(range(NUM_CLS)))
        fig, ax = plt.subplots(figsize=(8,6))
        import seaborn as sns
        sns.heatmap(cm, annot=True, fmt='d', xticklabels=LABELS, yticklabels=LABELS,
                    cmap='Blues', ax=ax)
        ax.set_title(f"{k.upper()} Confusion Matrix"); ax.set_ylabel("True"); ax.set_xlabel("Pred")
        plt.tight_layout(); fig.savefig(pd_dir/f"cm_{k}.png", dpi=200); plt.close()

    # 4. Per-class F1
    fig, ax = plt.subplots(figsize=(12,5))
    x = np.arange(NUM_CLS); w = 0.25
    for i, k in enumerate(results):
        if "test_f1_cls" in results[k]:
            ax.bar(x + i*w, results[k]["test_f1_cls"], w, label=k.upper())
    ax.set_xticks(x+w); ax.set_xticklabels(LABELS, rotation=30, ha='right')
    ax.set_ylabel("F1"); ax.set_title("Per-class F1 by Model"); ax.legend()
    plt.tight_layout(); fig.savefig(pd_dir/"per_class_f1.png", dpi=200); plt.close()

    # 5. Fed convergence
    fig, ax = plt.subplots(figsize=(10,5))
    for k in results:
        if "fed_hist" in results[k]:
            fh = results[k]["fed_hist"]
            ax.plot(fh["rounds"], fh["f1"], '-o', label=k.upper(), linewidth=2)
    ax.set_xlabel("Round"); ax.set_ylabel("F1"); ax.set_title("Federated Convergence")
    ax.legend(); plt.tight_layout(); fig.savefig(pd_dir/"fed_convergence.png", dpi=200); plt.close()
    print(f"  Plots saved to {pd_dir}")

# ═══════════════════════════════════════════════════════════════════════════════
# TEA LITERATURE COMPARISON  (32 papers from Tea Literature/ folder)
# ═══════════════════════════════════════════════════════════════════════════════
RESEARCH_PAPERS = {
    "NNE-Tea (Karmokar 2015)":       {"f1": 0.910, "acc": 0.910, "cat": "Traditional ML",     "yr": 2015, "pm": 0.03},
    "SVM-Tea (Hossain 2018)":        {"f1": 0.913, "acc": 0.913, "cat": "Traditional ML",     "yr": 2018, "pm": 0.01},
    "NSGA-SVM (Mukhopadhyay 2020)":  {"f1": 0.830, "acc": 0.830, "cat": "Traditional ML",     "yr": 2020, "pm": 0.01},
    "CNN-4cls (Biswas 2018)":        {"f1": 0.959, "acc": 0.959, "cat": "Plant Disease CNN",  "yr": 2018, "pm": 5.0},
    "DepthSepCNN (Hu 2019)":         {"f1": 0.976, "acc": 0.976, "cat": "Plant Disease CNN",  "yr": 2019, "pm": 1.2},
    "LeNet-5 Tea (Gayathri 2020)":   {"f1": 0.902, "acc": 0.902, "cat": "Plant Disease CNN",  "yr": 2020, "pm": 0.06},
    "DL-Tea (Somnath 2021)":         {"f1": 0.945, "acc": 0.945, "cat": "Plant Disease CNN",  "yr": 2021, "pm": 12.0},
    "AX-RetinaNet (Bao 2022)":       {"f1": 0.954, "acc": 0.938, "cat": "Plant Disease CNN",  "yr": 2022, "pm": 38.0},
    "CNN-7cls (Singh 2022)":         {"f1": 0.845, "acc": 0.845, "cat": "Plant Disease CNN",  "yr": 2022, "pm": 2.0},
    "AutoDetect-8cls (2022)":        {"f1": 0.945, "acc": 0.945, "cat": "Plant Disease CNN",  "yr": 2022, "pm": 4.0},
    "DNN-Tea (Datta 2023)":          {"f1": 0.930, "acc": 0.930, "cat": "Plant Disease CNN",  "yr": 2023, "pm": 3.5},
    "MobileNetV2-6cls (Barai 2024)": {"f1": 0.946, "acc": 0.946, "cat": "Plant Disease CNN",  "yr": 2024, "pm": 3.4},
    "NASNet Tea (Jayanti 2024)":     {"f1": 0.920, "acc": 0.920, "cat": "Plant Disease CNN",  "yr": 2024, "pm": 5.3},
    "CNN-BD (Rahman 2024)":          {"f1": 0.966, "acc": 0.967, "cat": "Plant Disease CNN",  "yr": 2024, "pm": 0.28},
    "HybridPool-CNN (2024)":         {"f1": 0.925, "acc": 0.925, "cat": "Plant Disease CNN",  "yr": 2024, "pm": 4.5},
    "AttentionCNN (2024)":           {"f1": 0.952, "acc": 0.955, "cat": "Plant Disease CNN",  "yr": 2024, "pm": 8.5},
    "MobileNetV3-Tea (Pan 2024)":    {"f1": 0.955, "acc": 0.958, "cat": "Plant Disease CNN",  "yr": 2024, "pm": 5.4},
    "CropProt-DL (2025)":            {"f1": 0.960, "acc": 0.965, "cat": "Plant Disease CNN",  "yr": 2025, "pm": 10.0},
    "NeuroCNN-Tea (2025)":           {"f1": 0.975, "acc": 0.978, "cat": "Plant Disease CNN",  "yr": 2025, "pm": 15.0},
    "ProcCS-Tea (2025)":             {"f1": 0.935, "acc": 0.940, "cat": "Plant Disease CNN",  "yr": 2025, "pm": 3.0},
    "SmartAgri-Tea (2025)":          {"f1": 0.968, "acc": 0.970, "cat": "Plant Disease CNN",  "yr": 2025, "pm": 7.0},
    "ResidualCNN (Rahat 2025)":      {"f1": 0.990, "acc": 0.990, "cat": "Plant Disease CNN",  "yr": 2025, "pm": 8.0},
    "YOLO-Tea (Xue 2023)":          {"f1": 0.920, "acc": 0.920, "cat": "Object Detection",   "yr": 2023, "pm": 7.2},
    "YOLO-T (Soebi 2023)":          {"f1": 0.965, "acc": 0.982, "cat": "Object Detection",   "yr": 2023, "pm": 36.9},
    "IntegratedEns (Wang 2023)":     {"f1": 0.793, "acc": 0.793, "cat": "Object Detection",   "yr": 2023, "pm": 12.0},
    "TL-TLB (Yao 2024)":            {"f1": 0.888, "acc": 0.922, "cat": "Object Detection",   "yr": 2024, "pm": 6.2},
    "FedCNN-Severity (Vats 2024)":   {"f1": 0.950, "acc": 0.950, "cat": "Federated Learning", "yr": 2024, "pm": 2.0},
    "FL-Sunflower (Alam 2024)":      {"f1": 0.904, "acc": 0.949, "cat": "Federated Learning", "yr": 2024, "pm": 3.5},
    "FedAvg (McMahan 2017)":         {"f1": 0.720, "acc": 0.750, "cat": "Federated Learning", "yr": 2017, "pm": 5.2},
    "FedProx (Li 2020)":             {"f1": 0.740, "acc": 0.770, "cat": "Federated Learning", "yr": 2020, "pm": 5.4},
}

def save_literature_plots(results, out_dir):
    """Generate comparison plots: FarmFederate vs 32 Tea Literature papers."""
    pd_dir = Path(out_dir) / "plots"; pd_dir.mkdir(parents=True, exist_ok=True)
    try: plt.style.use('seaborn-v0_8-whitegrid')
    except: plt.style.use('ggplot')

    # Best FarmFederate F1 (take best of VLM cent, VLM fed, ViT cent, etc.)
    best_key = max(results, key=lambda k: max(results[k]["cent_f1"], results[k]["fed_f1"]))
    best_f1 = max(results[best_key]["cent_f1"], results[best_key]["fed_f1"])
    best_acc = results[best_key]["cent_acc"]

    # ── Plot L1: F1 comparison bar chart (sorted) ────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 10))
    all_papers = {k: v["f1"] for k, v in RESEARCH_PAPERS.items() if v["f1"] is not None}
    all_papers["FarmFederate (Ours)"] = best_f1
    sorted_papers = sorted(all_papers.items(), key=lambda x: x[1])
    names = [p[0] for p in sorted_papers]
    scores = [p[1] for p in sorted_papers]
    colors = ['#e74c3c' if n == "FarmFederate (Ours)" else
              '#3498db' if RESEARCH_PAPERS.get(n, {}).get("cat") == "Plant Disease CNN" else
              '#2ecc71' if RESEARCH_PAPERS.get(n, {}).get("cat") == "Object Detection" else
              '#f39c12' if RESEARCH_PAPERS.get(n, {}).get("cat") == "Federated Learning" else
              '#9b59b6' for n in names]
    bars = ax.barh(names, scores, color=colors, edgecolor='black', linewidth=0.5)
    # Highlight ours
    for bar, name in zip(bars, names):
        if name == "FarmFederate (Ours)":
            bar.set_linewidth(2); bar.set_edgecolor('#c0392b')
    ax.set_xlabel("F1 Score", fontsize=12); ax.set_xlim(0.6, 1.02)
    ax.set_title("Tea Disease Detection: FarmFederate vs Literature (32 papers)", fontsize=13)
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#e74c3c', label='FarmFederate (Ours)'),
                       Patch(facecolor='#3498db', label='Plant Disease CNN'),
                       Patch(facecolor='#2ecc71', label='Object Detection'),
                       Patch(facecolor='#f39c12', label='Federated Learning'),
                       Patch(facecolor='#9b59b6', label='Traditional ML')]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
    for i, (n, s) in enumerate(zip(names, scores)):
        ax.text(s + 0.003, i, f"{s:.3f}", va='center', fontsize=7)
    plt.tight_layout(); fig.savefig(pd_dir / "lit_f1_comparison.png", dpi=250); plt.close()
    print("  [L1] Literature F1 comparison saved")

    # ── Plot L2: Accuracy vs Year scatter ────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 7))
    cat_colors = {"Traditional ML": "#9b59b6", "Plant Disease CNN": "#3498db",
                  "Object Detection": "#2ecc71", "Federated Learning": "#f39c12"}
    for name, data in RESEARCH_PAPERS.items():
        if data["acc"] is None: continue
        c = cat_colors.get(data["cat"], "#95a5a6")
        ax.scatter(data["yr"], data["acc"], c=c, s=max(20, data["pm"]*8),
                   edgecolors='black', linewidth=0.5, alpha=0.7, zorder=2)
    # Plot ours as a star
    ax.scatter(2025, best_acc, c='#e74c3c', s=300, marker='*',
               edgecolors='black', linewidth=1, zorder=5, label='FarmFederate (Ours)')
    ax.set_xlabel("Year", fontsize=12); ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title("Tea Disease Detection: Accuracy vs Year (bubble = model size)", fontsize=13)
    ax.legend(fontsize=9); ax.set_ylim(0.7, 1.02)
    plt.tight_layout(); fig.savefig(pd_dir / "lit_accuracy_vs_year.png", dpi=250); plt.close()
    print("  [L2] Accuracy vs Year scatter saved")

    # ── Plot L3: F1 vs Model Size bubble ─────────────────────────────────────
    fig, ax = plt.subplots(figsize=(12, 7))
    for name, data in RESEARCH_PAPERS.items():
        if data["f1"] is None or data["pm"] is None: continue
        c = cat_colors.get(data["cat"], "#95a5a6")
        ax.scatter(data["pm"], data["f1"], c=c, s=100, edgecolors='black',
                   linewidth=0.5, alpha=0.7, zorder=2)
        ax.annotate(name.split("(")[0].strip(), (data["pm"], data["f1"]),
                    fontsize=5, ha='center', va='bottom')
    # Ours — use approximate param count for VLM
    our_params = 4.5  # approx M params for VLM concat-fusion
    ax.scatter(our_params, best_f1, c='#e74c3c', s=250, marker='*',
               edgecolors='black', linewidth=1.5, zorder=5)
    ax.annotate("FarmFederate\n(Ours)", (our_params, best_f1),
                fontsize=8, fontweight='bold', ha='center', va='bottom', color='#c0392b')
    ax.set_xlabel("Parameters (Millions)", fontsize=12)
    ax.set_ylabel("F1 Score", fontsize=12)
    ax.set_title("F1 vs Model Complexity — FarmFederate vs Literature", fontsize=13)
    ax.set_ylim(0.7, 1.02)
    plt.tight_layout(); fig.savefig(pd_dir / "lit_f1_vs_params.png", dpi=250); plt.close()
    print("  [L3] F1 vs Params bubble saved")

    # ── Plot L4: Category box plot ───────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 6))
    cats = ["Traditional ML", "Plant Disease CNN", "Object Detection", "Federated Learning"]
    cat_f1s = {c: [] for c in cats}
    for data in RESEARCH_PAPERS.values():
        if data["f1"] is not None and data["cat"] in cat_f1s:
            cat_f1s[data["cat"]].append(data["f1"])
    box_data = [cat_f1s[c] for c in cats]
    bp = ax.boxplot(box_data, labels=cats, patch_artist=True, widths=0.5)
    for patch, c in zip(bp['boxes'], [cat_colors[c] for c in cats]):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    # Our line
    ax.axhline(y=best_f1, color='#e74c3c', linestyle='--', linewidth=2,
               label=f'FarmFederate (Ours): {best_f1:.3f}')
    ax.set_ylabel("F1 Score", fontsize=12)
    ax.set_title("F1 Distribution by Method Category vs FarmFederate", fontsize=13)
    ax.legend(fontsize=10); ax.set_ylim(0.6, 1.05)
    plt.tight_layout(); fig.savefig(pd_dir / "lit_category_boxplot.png", dpi=250); plt.close()
    print("  [L4] Category box plot saved")

    # ── Print comparison table ───────────────────────────────────────────────
    print(f"\n  {'='*75}")
    print(f"  TEA LITERATURE COMPARISON (32 papers)")
    print(f"  {'='*75}")
    print(f"  {'Paper':<35} {'F1':>6} {'Acc':>6} {'Year':>5} {'Category':<20}")
    print(f"  {'-'*75}")
    for name, data in sorted(RESEARCH_PAPERS.items(), key=lambda x: x[1].get("f1") or 0, reverse=True):
        f1s = f"{data['f1']:.3f}" if data['f1'] else "  N/A"
        acs = f"{data['acc']:.3f}" if data['acc'] else "  N/A"
        print(f"  {name:<35} {f1s:>6} {acs:>6} {data['yr']:>5} {data['cat']:<20}")
    print(f"  {'-'*75}")
    print(f"  {'FarmFederate (Ours)':<35} {best_f1:>6.3f} {best_acc:>6.3f} {'2025':>5} {'Multimodal FL':<20}")
    print(f"  {'='*75}")

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════
def main():
    # Colab Check & Drive Mounting
    in_colab = False
    try:
        import google.colab
        in_colab = True
        print("\n  [Colab Environment Detected] Mounting Google Drive...")
        google.colab.drive.mount('/content/drive')
    except ImportError:
        pass

    ap = argparse.ArgumentParser(description="Tea Disease - Real Dataset - FarmFederate")
    if in_colab:
        ap.add_argument("--data_dir", default="/content/drive/MyDrive/FarmFederate/Real Dataset")
        ap.add_argument("--out_dir", default="/content/drive/MyDrive/FarmFederate/tea_results")
    else:
        ap.add_argument("--data_dir", default="C:/Users/USER_HP/Desktop/FarmFederate/Real Dataset")
        ap.add_argument("--out_dir", default="C:/Users/USER_HP/Desktop/FarmFederate/tea_results")
    ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--fed_rounds", type=int, default=8)
    ap.add_argument("--num_clients", type=int, default=3)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    cfg = Cfg(epochs=args.epochs, fed_rounds=args.fed_rounds, num_clients=args.num_clients,
              lr=args.lr, seed=args.seed,
              data_dir=args.data_dir, out_dir=args.out_dir)

    random.seed(cfg.seed); np.random.seed(cfg.seed); torch.manual_seed(cfg.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"\n{'='*70}")
    print("FARMFEDERATE — TEA LEAF DISEASE DETECTION (Real Dataset Only)")
    print(f"{'='*70}")
    print(f"  Device  : {dev}")
    print(f"  Classes : {LABELS}")
    print(f"  Source  : {cfg.data_dir}  (ONLY source — no external data)")
    print(f"  Epochs  : {cfg.epochs}  |  Fed rounds: {cfg.fed_rounds}  |  Clients: {cfg.num_clients}")

    # ── PHASE 1: Extract & classify crops from Real Dataset ──────────────────
    print(f"\n{'─'*70}")
    print("PHASE 1: Extract & classify disease crops from OBB labels")
    print(f"{'─'*70}")
    img_dir, lbl_dir = str(Path(cfg.data_dir)/"images"), str(Path(cfg.data_dir)/"labels")
    full = OBBDS(img_dir, lbl_dir, val_tf(cfg.img_size), cfg.crop_pad)
    al = full.labels
    print(f"  Extracted {len(full)} disease crops from OBB annotations")
    for i, n in enumerate(LABELS): print(f"    [{i}] {n:<22}: {Counter(al).get(i,0):>4} crops")

    # ── PHASE 2: Generate text descriptions per crop (for LLM) ──────────────
    print(f"\n{'─'*70}")
    print("PHASE 2: Generate text descriptions per crop (literature-grounded)")
    print(f"{'─'*70}")
    tdf = gen_crop_descriptions(full.samples, seed=cfg.seed)
    print(f"  Generated {len(tdf)} text descriptions (1 per crop)")
    print(f"  Distribution: {dict(Counter(tdf['label_name']))}")
    print(f"  Sample text: {tdf.iloc[0]['text'][:100]}...")

    # ── PHASE 3: Stratified split (80/10/10) on unified dataset ─────────────
    print(f"\n{'─'*70}")
    print("PHASE 3: Stratified split (same indices for image + text)")
    print(f"{'─'*70}")
    ci = {i: [] for i in range(NUM_CLS)}
    for j, l in enumerate(al): ci[l].append(j)
    tr_i, va_i, te_i = [], [], []
    rng = random.Random(cfg.seed)
    for _, idxs in ci.items():
        s = idxs.copy(); rng.shuffle(s)
        nt = max(1, int(len(s)*cfg.train_split)); nv = max(1, int(len(s)*cfg.val_split))
        tr_i.extend(s[:nt]); va_i.extend(s[nt:nt+nv]); te_i.extend(s[nt+nv:])
    tr_lbl = [al[i] for i in tr_i]
    print(f"  train={len(tr_i)}  val={len(va_i)}  test={len(te_i)}")
    print(f"  Train distribution: {dict(Counter(tr_lbl))}")

    cw = class_weights(tr_lbl, NUM_CLS).to(dev)
    print(f"  Class weights: {cw.cpu().numpy().round(3).tolist()}")

    # Text split uses SAME indices as image split (unified dataset)
    df_tr = tdf.iloc[tr_i].reset_index(drop=True)
    df_va = tdf.iloc[va_i].reset_index(drop=True)
    df_te = tdf.iloc[te_i].reset_index(drop=True)
    tl_tr = df_tr["labels"].tolist()

    # ── PHASE 4: Build datasets ─────────────────────────────────────────────
    print(f"\n{'─'*70}")
    print("PHASE 4: Build ViT (image), LLM (text), VLM (multimodal) datasets")
    print(f"{'─'*70}")
    vit_tr = OBBDS(img_dir, lbl_dir, aug_tf(cfg.img_size), cfg.crop_pad, tr_i)
    vit_va = OBBDS(img_dir, lbl_dir, val_tf(cfg.img_size), cfg.crop_pad, va_i)
    vit_te = OBBDS(img_dir, lbl_dir, val_tf(cfg.img_size), cfg.crop_pad, te_i)
    print(f"  ViT datasets  : train={len(vit_tr)} val={len(vit_va)} test={len(vit_te)}")

    llm_tr, llm_va, llm_te = TextDS(df_tr), TextDS(df_va), TextDS(df_te)
    print(f"  LLM datasets  : train={len(llm_tr)} val={len(llm_va)} test={len(llm_te)}")

    vlm_tr = MMDS(vit_tr, df_tr, aug_tf(cfg.img_size), seed=cfg.seed)
    vlm_va = MMDS(vit_va, df_va, val_tf(cfg.img_size), seed=cfg.seed)
    vlm_te = MMDS(vit_te, df_te, val_tf(cfg.img_size), seed=cfg.seed)
    print(f"  VLM datasets  : train={len(vlm_tr)} val={len(vlm_va)} test={len(vlm_te)}")

    # ── Loaders ──────────────────────────────────────────────────────────────
    vit_trl = DataLoader(vit_tr, batch_sampler=BalSampler(tr_lbl, cfg.batch_size, NUM_CLS))
    vit_val = DataLoader(vit_va, batch_size=cfg.batch_size)
    vit_tel = DataLoader(vit_te, batch_size=cfg.batch_size)
    llm_trl = DataLoader(llm_tr, batch_sampler=BalSampler(tl_tr, cfg.batch_size, NUM_CLS))
    llm_val = DataLoader(llm_va, batch_size=cfg.batch_size)
    llm_tel = DataLoader(llm_te, batch_size=cfg.batch_size)
    vlm_trl = DataLoader(vlm_tr, batch_sampler=BalSampler(tr_lbl, cfg.batch_size, NUM_CLS))
    vlm_val = DataLoader(vlm_va, batch_size=cfg.batch_size)
    vlm_tel = DataLoader(vlm_te, batch_size=cfg.batch_size)

    configs = [
        ("llm","text",LLM,dict(nc=NUM_CLS), llm_trl,llm_val,llm_tel, llm_tr),
        ("vit","vision",ViT,dict(nc=NUM_CLS,cw=cw,focal=True), vit_trl,vit_val,vit_tel, vit_tr),
        ("vlm","multimodal",VLM,dict(nc=NUM_CLS,cw=cw,focal=True), vlm_trl,vlm_val,vlm_tel, vlm_tr),
    ]

    results = {}; models_dir = Path(cfg.out_dir)/"models"; models_dir.mkdir(parents=True, exist_ok=True)

    for key, mt, mcls, mkw, trl, val, tel, trds in configs:
        print(f"\n{'='*70}\nCENTRALISED — {key.upper()} ({mt})\n{'='*70}")
        model = mcls(**mkw).to(dev)
        bf1, hist, bst = train_model(model, trl, val, cfg, dev, mt, models_dir=models_dir, prefix=key)
        if bst: model.load_state_dict(bst)
        torch.save({"state_dict": model.state_dict(), "class": mcls.__name__,
                     "kwargs": mkw, "labels": LABELS}, models_dir/f"best_cent_{key}.pt")
        te = evaluate(model, tel, dev, mt)
        print(f"\n  [{key.upper()}] Test F1={te['f1']:.4f} F1-macro={te['f1_macro']:.4f} Acc={te['acc']:.4f}")
        for i,(n,v) in enumerate(zip(LABELS, te["f1_cls"])): print(f"    [{i}] {n:<22}: {v:.4f}")
        print(f"\n  Classification Report:\n{classification_report(te['labels'],te['preds'],target_names=LABELS,zero_division=0)}")

        print(f"\n{'─'*70}\nFEDERATED — {key.upper()} ({cfg.num_clients} clients, {cfg.fed_rounds} rounds)\n{'─'*70}")
        ff1, fh = federated_train(mcls, mkw, trds, val, cfg, dev, mt, models_dir=models_dir, prefix=key)
        print(f"  Best Fed F1={ff1:.4f}")

        results[key] = {"cent_f1": te["f1"], "cent_f1_macro": te["f1_macro"], "cent_acc": te["acc"],
                        "fed_f1": ff1, "history": hist, "fed_hist": fh,
                        "test_preds": te["preds"], "test_labels": te["labels"],
                        "test_f1_cls": te["f1_cls"]}

    # Plots
    save_plots(results, cfg.out_dir)

    # Literature comparison (32 papers from Tea Literature/)
    print(f"\n{'─'*70}")
    print("LITERATURE COMPARISON — FarmFederate vs 32 Tea Disease Papers")
    print(f"{'─'*70}")
    save_literature_plots(results, cfg.out_dir)

    # Summary
    print(f"\n{'='*70}\nFINAL SUMMARY\n{'='*70}")
    print(f"  {'Model':<20} {'Cent F1':>10} {'Fed F1':>10} {'Test Acc':>10}")
    print(f"  {'-'*52}")
    for k, mt in [("llm","LLM (text)"),("vit","ViT (vision)"),("vlm","VLM (multimodal)")]:
        r = results[k]
        print(f"  {mt:<20} {r['cent_f1']:>10.4f} {r['fed_f1']:>10.4f} {r['cent_acc']:>10.4f}")

    # Save JSON
    json_out = {k: {kk: (v.tolist() if isinstance(v, np.ndarray) else v)
                     for kk, v in vv.items() if kk not in ("test_preds","test_labels")}
                for k, vv in results.items()}
    with open(Path(cfg.out_dir)/"results.json", "w") as f:
        json.dump(json_out, f, indent=2, default=str)
    print(f"\n  Results saved to {cfg.out_dir}")

    # ── PHASE 5: Remedy Recommendations (Inference Demo) ────────────────────
    print(f"\n{'='*70}")
    print("PHASE 5: Disease Detection & Remedy Recommendation generated")
    print(f"{'='*70}")
    remedy_report = []
    # Test on a few samples from VLM test set
    vlm_model = VLM(nc=NUM_CLS, cw=cw, focal=True).to(dev)
    try:
        st = torch.load(models_dir/"best_cent_vlm.pt", map_location=dev)["state_dict"]
        vlm_model.load_state_dict(st)
        vlm_model.eval()
    except Exception as e:
        print(f"  Warning: Could not load VLM for recommendations: {e}")
        
    rng_test = random.Random(cfg.seed)
    test_indices = rng_test.sample(range(len(vlm_te)), min(5, len(vlm_te)))
    print("  Sample Crop Analysis & Remedies:")
    for idx in test_indices:
        b = vlm_te[idx]
        with torch.no_grad():
            o = vlm_model(input_ids=b['input_ids'].unsqueeze(0).to(dev),
                          attention_mask=b['attention_mask'].unsqueeze(0).to(dev),
                          pixel_values=b['pixel_values'].unsqueeze(0).to(dev))
            pred_id = int(o['logits'].argmax(-1).item())
            true_id = b['labels'].argmax().item()
            
        pred_disease = LABELS[pred_id]
        true_disease = LABELS[true_id]
        crop_text = _tok.te_decode(b['input_ids']) if hasattr(_tok, 'te_decode') else "Crop text used for fusion."
        remedy = REMEDY_RECOMMENDATIONS[pred_disease]
        
        rep = {
            "crop_idx": idx,
            "true_disease": true_disease,
            "predicted_disease": pred_disease,
            "remedy": remedy
        }
        remedy_report.append(rep)
        
        print(f"\n  Crop {idx}:")
        print(f"  - Actual  : {true_disease}")
        print(f"  - Predict : {pred_disease}")
        print(f"  - Remedy  : {remedy}")

    with open(Path(cfg.out_dir)/"remedy_report.json", "w") as f:
        json.dump(remedy_report, f, indent=2)
    print(f"\n  Remedy report saved to {cfg.out_dir}/remedy_report.json")

if __name__ == "__main__":
    main()

